## Task 1: Conceptual Understanding

### 1. What is the difference between "Love" and "love" in NLP?
To a human, "Love" and "love" mean the exact same thing. However, to a computer and standard NLP algorithms, they are treated as **two completely distinct tokens**.

Because algorithms read text as numerical representations (like ASCII or Unicode values), the uppercase "L" makes "Love" a different data point than the lowercase "l" in "love". If the text is not normalized by converting everything to lowercase, it artificially inflates the model's vocabulary size and splits the frequency counts of the word. This makes it harder for the model to recognize that both words carry the same semantic weight.

### 2. What happens if stopwords are not removed?
Stopwords (like "the", "is", "in", "and") are the most frequently occurring words in a language. If left in the dataset:
* **Computational bloat:** They drastically increase vocabulary size, requiring more memory and processing power.
* **Statistical noise:** In models relying on word frequency (like Bag-of-Words or TF-IDF), stopwords dominate the dataset, drowning out the rarer, domain-specific keywords that carry the core meaning.
* **Diluted focus:** The model may spend unnecessary resources learning patterns around conjunctions and prepositions rather than focusing on the actual subject matter.

### 3. Two real-world scenarios where removing stopwords can be harmful
While removing stopwords is standard for tasks like topic modeling, it can be destructive when syntax and sequence matter:
* **Sentiment Analysis:** Many standard stopword lists include negation words like "not", "no", or "nor". If removed, a sentence like *"I am not happy with this product"* becomes *"I happy product"*, which entirely flips the sentiment from negative to positive.
* **Machine Translation & Text Generation:** Generative models and translation systems rely heavily on grammar, syntax, and phrasing to produce human-readable text. Removing conjunctions and prepositions destroys the structural context necessary to generate coherent sentences.

### 4. Stemming vs. Lemmatization
Both techniques aim to reduce words to their base forms, but they do it in fundamentally different ways:
* **Stemming is a crude, rule-based approach.** It simply chops off prefixes or suffixes (like "-ing", "-ed", "-s") based on predefined rules. It is very fast, but it lacks contextual understanding, often producing "stems" that are not actual English words. *(Example: "Caring" -> "Car")*
* **Lemmatization is a sophisticated, dictionary-based approach.** It analyzes the word's morphology and context (its part of speech) to return its actual dictionary root, known as the "lemma." It is computationally heavier, but the output is always a valid word. *(Example: "Better" -> "Good")*

Task 2: The Preprocessing Function

In [11]:
import re

def preprocess_text(text):
    # 1. Convert text to lowercase
    text = text.lower()

    # 2. Remove URLs and email-like patterns
    text = re.sub(r'http[s]?://\S+|www\.\S+|\S+@\S+', '', text)

    # 3. Remove numbers
    text = re.sub(r'\d+', '', text)

    # 4. Handle repeated characters
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    text = re.sub(r'\bsoo\b', 'so', text)

    # 5. Remove extra spaces
    tokens = text.split()

    # 6. Remove very short tokens
    exceptions = {'no', 'not', 'i', 'is', 'so'}
    cleaned_tokens = [word for word in tokens if len(word) > 2 or word in exceptions]

    cleaned_sentence = ' '.join(cleaned_tokens)
    return cleaned_sentence

# Output Generation
test_cases = [
    "I have 2 dogs",
    "This is  good",
    "soooo goooood!!!",
    "WOW!!! This is GREAT!!!",
    "Visit http://example.com now",
    "no this is not ok"
]

for sentence in test_cases:
    print(f"Original : '{sentence}'")
    print(f"Cleaned  : '{preprocess_text(sentence)}'")
    print("-" * 40)

Original : 'I have 2 dogs'
Cleaned  : 'i have dogs'
----------------------------------------
Original : 'This is  good'
Cleaned  : 'this is good'
----------------------------------------
Original : 'soooo goooood!!!'
Cleaned  : 'so good'
----------------------------------------
Original : 'WOW!!! This is GREAT!!!'
Cleaned  : 'wow this is great'
----------------------------------------
Original : 'Visit http://example.com now'
Cleaned  : 'visit now'
----------------------------------------
Original : 'no this is not ok'
Cleaned  : 'no this is not'
----------------------------------------


Task 3: Stress Testing

In [12]:
import re

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'http[s]?://\S+|www\.\S+|\S+@\S+', '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    text = re.sub(r'\bsoo\b', 'so', text)

    tokens = text.split()

    exceptions = {'no', 'not', 'ok', 'i', 'is', 'so', 'me', 'at'}
    cleaned_tokens = [word for word in tokens if len(word) > 2 or word in exceptions]
    cleaned_sentence = ' '.join(cleaned_tokens)

    # Returning both tokens and sentence to satisfy Task 3 output requirements
    return cleaned_tokens, cleaned_sentence


# ==========================================
# Task 3: Stress Testing
# ==========================================
sample_inputs = [
    "Get 100% FREE access now!!!",
    "I absolutely looooved this product 😍😍",
    "Worst service ever... 0/10",
    "Call me at 9876543210",
    "This is THE best course!!!",
    "Visit https://openai.com now!",
    "Nooooo this is baaad!!!",
    "OK OK OK I got it",
    "Win $$$ now!!! Limited offer!!!",
    "I am not happy with this"
]

print("--- Task 3: Stress Testing ---\n")

for text in sample_inputs:
    tokens, sentence = preprocess_text(text)
    print(f"Original Text    : {text}")
    print(f"Cleaned Tokens   : {tokens}")
    print(f"Cleaned Sentence : {sentence}")
    print("-" * 50)

--- Task 3: Stress Testing ---

Original Text    : Get 100% FREE access now!!!
Cleaned Tokens   : ['get', 'free', 'access', 'now']
Cleaned Sentence : get free access now
--------------------------------------------------
Original Text    : I absolutely looooved this product 😍😍
Cleaned Tokens   : ['i', 'absolutely', 'looved', 'this', 'product']
Cleaned Sentence : i absolutely looved this product
--------------------------------------------------
Original Text    : Worst service ever... 0/10
Cleaned Tokens   : ['worst', 'service', 'ever']
Cleaned Sentence : worst service ever
--------------------------------------------------
Original Text    : Call me at 9876543210
Cleaned Tokens   : ['call', 'me', 'at']
Cleaned Sentence : call me at
--------------------------------------------------
Original Text    : This is THE best course!!!
Cleaned Tokens   : ['this', 'is', 'the', 'best', 'course']
Cleaned Sentence : this is the best course
--------------------------------------------------
Origina

Task 4: Token Analytics

In [13]:
print("--- Token-Level Statistical Analysis ---\n")

for text in sample_inputs:
    # Get the cleaned tokens from our previous function
    tokens, sentence = preprocess_text(text)

    # 1. Total number of tokens
    total_tokens = len(tokens)

    # 2. Number of unique tokens (using a Set removes duplicates)
    unique_tokens = len(set(tokens))

    # 3. Average token length
    if total_tokens > 0:
        avg_length = sum(len(word) for word in tokens) / total_tokens
    else:
        avg_length = 0

    # Print the results
    print(f"Original       : '{text}'")
    print(f"Cleaned Tokens : {tokens}")
    print(f"Total Tokens   : {total_tokens}")
    print(f"Unique Tokens  : {unique_tokens}")
    print(f"Avg Length     : {avg_length:.2f} characters")
    print("-" * 50)

--- Token-Level Statistical Analysis ---

Original       : 'Get 100% FREE access now!!!'
Cleaned Tokens : ['get', 'free', 'access', 'now']
Total Tokens   : 4
Unique Tokens  : 4
Avg Length     : 4.00 characters
--------------------------------------------------
Original       : 'I absolutely looooved this product 😍😍'
Cleaned Tokens : ['i', 'absolutely', 'looved', 'this', 'product']
Total Tokens   : 5
Unique Tokens  : 5
Avg Length     : 5.60 characters
--------------------------------------------------
Original       : 'Worst service ever... 0/10'
Cleaned Tokens : ['worst', 'service', 'ever']
Total Tokens   : 3
Unique Tokens  : 3
Avg Length     : 5.33 characters
--------------------------------------------------
Original       : 'Call me at 9876543210'
Cleaned Tokens : ['call', 'me', 'at']
Total Tokens   : 3
Unique Tokens  : 3
Avg Length     : 2.67 characters
--------------------------------------------------
Original       : 'This is THE best course!!!'
Cleaned Tokens : ['this', 'is', '

Task 5: Frequency Analysis


In [14]:
from collections import Counter

print("--- Task 5: Frequency Analysis ---\n")

# 1. Combine all tokens from all sentences into a single, flat list
all_tokens = []
for text in sample_inputs:
    tokens, _ = preprocess_text(text)
    all_tokens.extend(tokens)  # extend adds elements of the list, not the list itself

# 2. Count the frequencies of every word
word_counts = Counter(all_tokens)

# 3. Identify Top 10 most frequent words
top_10 = word_counts.most_common(10)

# 4. Identify Top 5 least frequent words
least_5 = word_counts.most_common()[-5:]
least_5.reverse()

print(f"Total vocabulary pool: {len(all_tokens)} tokens\n")

print("🏆 Top 10 Most Frequent Words:")
for i, (word, count) in enumerate(top_10, 1):
    print(f"  {i}. '{word}' (appears {count} times)")

print("\n🔍 Top 5 Least Frequent Words:")
for i, (word, count) in enumerate(least_5, 1):
    print(f"  {i}. '{word}' (appears {count} times)")

--- Task 5: Frequency Analysis ---

Total vocabulary pool: 40 tokens

🏆 Top 10 Most Frequent Words:
  1. 'this' (appears 4 times)
  2. 'now' (appears 3 times)
  3. 'i' (appears 3 times)
  4. 'ok' (appears 3 times)
  5. 'is' (appears 2 times)
  6. 'get' (appears 1 times)
  7. 'free' (appears 1 times)
  8. 'access' (appears 1 times)
  9. 'absolutely' (appears 1 times)
  10. 'looved' (appears 1 times)

🔍 Top 5 Least Frequent Words:
  1. 'with' (appears 1 times)
  2. 'happy' (appears 1 times)
  3. 'not' (appears 1 times)
  4. 'offer' (appears 1 times)
  5. 'limited' (appears 1 times)


Task 6: Build Full Pipeline


In [18]:
import re
import json
def preprocess_text(text):
    if not isinstance(text, str): return [], ""
    text = text.lower()
    text = re.sub(r'http[s]?://\S+|www\.\S+|\S+@\S+', '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    text = re.sub(r'\bsoo\b', 'so', text)
    tokens = text.split()
    exceptions = {'no', 'not', 'ok', 'i', 'is', 'so', 'me', 'at'}
    cleaned_tokens = [word for word in tokens if len(word) > 2 or word in exceptions]
    return cleaned_tokens, ' '.join(cleaned_tokens)

# Task 6: Build Full Pipeline
def full_pipeline(text_list):
    pipeline_result = {
        "tokens": [],
        "clean_sentences": []
    }
    for text in text_list:
        tokens, sentence = preprocess_text(text)
        pipeline_result["tokens"].append(tokens)
        pipeline_result["clean_sentences"].append(sentence)
    return pipeline_result

# Task 6 Output (Using the 10 Sample Inputs)
sample_inputs = [
    "Get 100% FREE access now!!!",
    "I absolutely looooved this product 😍😍",
    "Worst service ever... 0/10",
    "Call me at 9876543210",
    "This is THE best course!!!",
    "Visit https://openai.com now!",
    "Nooooo this is baaad!!!",
    "OK OK OK I got it",
    "Win $$$ now!!! Limited offer!!!",
    "I am not happy with this"
]

print("=== TASK 6: PIPELINE OUTPUT ===")
task6_output = full_pipeline(sample_inputs)
print("{")
print('    "tokens": [')
for t in task6_output['tokens']:
    print(f"        {t},")
print('    ],')
print('    "clean_sentences": [')
for s in task6_output['clean_sentences']:
    print(f'        "{s}",')
print('    ]')
print("}")


=== TASK 6: PIPELINE OUTPUT ===
{
    "tokens": [
        ['get', 'free', 'access', 'now'],
        ['i', 'absolutely', 'looved', 'this', 'product'],
        ['worst', 'service', 'ever'],
        ['call', 'me', 'at'],
        ['this', 'is', 'the', 'best', 'course'],
        ['visit', 'now'],
        ['noo', 'this', 'is', 'baad'],
        ['ok', 'ok', 'ok', 'i', 'got'],
        ['win', 'now', 'limited', 'offer'],
        ['i', 'not', 'happy', 'with', 'this'],
    ],
    "clean_sentences": [
        "get free access now",
        "i absolutely looved this product",
        "worst service ever",
        "call me at",
        "this is the best course",
        "visit now",
        "noo this is baad",
        "ok ok ok i got",
        "win now limited offer",
        "i not happy with this",
    ]
}


Task 7: Error Handling

In [16]:
edge_cases = [
    "",             # Empty string
    "😍😍😍",        # Only emojis
    "1234567890"    # Only numbers
]

print("--- Task 7: Error Handling Test ---\n")

final_output = full_pipeline(edge_cases)

for i, raw_text in enumerate(edge_cases):
    print(f"Original Text    : {repr(raw_text)}")
    print(f"Cleaned Tokens   : {final_output['tokens'][i]}")
    print(f"Cleaned Sentence : {repr(final_output['clean_sentences'][i])}")
    print("-" * 40)

print("\nFinal Pipeline Output:")
print(final_output)

--- Task 7: Error Handling Test ---

Original Text    : ''
Cleaned Tokens   : []
Cleaned Sentence : ''
----------------------------------------
Original Text    : '😍😍😍'
Cleaned Tokens   : []
Cleaned Sentence : ''
----------------------------------------
Original Text    : '1234567890'
Cleaned Tokens   : []
Cleaned Sentence : ''
----------------------------------------

Final Pipeline Output:
{'tokens': [[], [], []], 'clean_sentences': ['', '', '']}
